In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:

from pyspark.sql import functions as F

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SOURCE_SYSTEM = "football-data.org"

BRONZE_TABLE = f"{BRONZE_SCHEMA}.football_data_org_matches"
SILVER_TABLE = f"{SILVER_SCHEMA}.football_data_org_matches"

# One row represents one football-data.org match.
BUSINESS_KEY = ["match_id"]

REQUIRED_SILVER_COLUMNS = [
    "match_id",
    "competition_id",
    "competition_code",
    "league_code",
    "season_id",
    "season_start_year",
    "match_utc_datetime",
    "match_date",
    "status",
    "home_team_id",
    "away_team_id",
    "home_team_key",
    "away_team_key",
]


# -----------------------------------------------------------------------------
# Read Bronze
# -----------------------------------------------------------------------------

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )

bronze_df = spark.table(BRONZE_TABLE)


# -----------------------------------------------------------------------------
# Validate required top-level Bronze columns
# -----------------------------------------------------------------------------

REQUIRED_BRONZE_COLUMNS = [
    "id",
    "utcDate",
    "status",
    "competition",
    "season",
    "homeTeam",
    "awayTeam",
    "score",
    "_league",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Helper for alternative score field names
# -----------------------------------------------------------------------------

def optional_first(paths, data_type="int"):
    for path in paths:
        if field_exists(bronze_df, path):
            return F.col(path).cast(data_type)

    return F.lit(None).cast(data_type)

# -----------------------------------------------------------------------------
# Validate required nested Bronze fields
# -----------------------------------------------------------------------------

REQUIRED_NESTED_COLUMNS = [
    "competition.id",
    "competition.code",
    "season.id",
    "season.startDate",
    "homeTeam.id",
    "awayTeam.id",
]

missing_nested_columns = [
    path
    for path in REQUIRED_NESTED_COLUMNS
    if not field_exists(bronze_df, path)
]

if missing_nested_columns:
    raise ValueError(
        "Required nested Bronze columns are missing: "
        f"{missing_nested_columns}"
    )


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate source identity
# -----------------------------------------------------------------------------

invalid_source_rows = (
    bronze_df
    .filter(
        F.col("_source_system").isNull()
        | (F.col("_source_system") != F.lit(SOURCE_SYSTEM))
    )
    .limit(1)
    .count()
)

if invalid_source_rows > 0:
    raise ValueError(
        f"Unexpected source system found in {BRONZE_TABLE}."
    )


# -----------------------------------------------------------------------------
# Select, rename and cast Silver columns
# -----------------------------------------------------------------------------

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Match identifier
        # ---------------------------------------------------------------------

        F.col("id")
        .cast("long")
        .alias("match_id"),


        # ---------------------------------------------------------------------
        # Competition
        # ---------------------------------------------------------------------

        F.col("competition.id")
        .cast("int")
        .alias("competition_id"),

        clean_string("competition.code")
        .alias("competition_code"),

        optional_string(bronze_df, "competition.name")
        .alias("competition_name"),

        clean_string("_league")
        .alias("league_code"),


        # ---------------------------------------------------------------------
        # Season
        # ---------------------------------------------------------------------

        F.col("season.id")
        .cast("int")
        .alias("season_id"),

        F.col("season.startDate")
        .cast("date")
        .alias("season_start_date"),

        optional_col(bronze_df, "season.endDate", "date")
        .alias("season_end_date"),


        # ---------------------------------------------------------------------
        # Match information
        # ---------------------------------------------------------------------

        F.to_timestamp(F.col("utcDate"))
        .alias("match_utc_datetime"),

        clean_string("status")
        .alias("status"),

        optional_col(bronze_df, "matchday", "int")
        .alias("matchday"),

        optional_string(bronze_df, "stage")
        .alias("stage"),

        optional_string(bronze_df, "group")
        .alias("match_group"),

        optional_string(bronze_df, "venue")
        .alias("venue"),

        optional_col(bronze_df, "minute", "int")
        .alias("match_minute"),

        optional_col(bronze_df, "injuryTime", "int")
        .alias("injury_time"),

        optional_col(bronze_df, "attendance", "int")
        .alias("attendance"),


        # ---------------------------------------------------------------------
        # Home team
        # ---------------------------------------------------------------------

        F.col("homeTeam.id")
        .cast("int")
        .alias("home_team_id"),

        optional_string(bronze_df, "homeTeam.name")
        .alias("home_team_name"),

        optional_string(bronze_df, "homeTeam.shortName")
        .alias("home_team_short_name"),

        optional_string(bronze_df, "homeTeam.tla")
        .alias("home_team_tla"),


        # ---------------------------------------------------------------------
        # Away team
        # ---------------------------------------------------------------------

        F.col("awayTeam.id")
        .cast("int")
        .alias("away_team_id"),

        optional_string(bronze_df, "awayTeam.name")
        .alias("away_team_name"),

        optional_string(bronze_df, "awayTeam.shortName")
        .alias("away_team_short_name"),

        optional_string(bronze_df, "awayTeam.tla")
        .alias("away_team_tla"),


        # ---------------------------------------------------------------------
        # Score summary
        # ---------------------------------------------------------------------

        optional_string(bronze_df, "score.winner")
        .alias("winner"),

        optional_string(bronze_df, "score.duration")
        .alias("score_duration"),


        # ---------------------------------------------------------------------
        # Full-time score
        # ---------------------------------------------------------------------

        optional_first([
            "score.fullTime.home",
            "score.fullTime.homeTeam",
        ])
        .alias("home_goals"),

        optional_first([
            "score.fullTime.away",
            "score.fullTime.awayTeam",
        ])
        .alias("away_goals"),


        # ---------------------------------------------------------------------
        # Half-time score
        # ---------------------------------------------------------------------

        optional_first([
            "score.halfTime.home",
            "score.halfTime.homeTeam",
        ])
        .alias("home_half_time_goals"),

        optional_first([
            "score.halfTime.away",
            "score.halfTime.awayTeam",
        ])
        .alias("away_half_time_goals"),


        # ---------------------------------------------------------------------
        # Regular-time score (90 minutes)
        # ---------------------------------------------------------------------

        optional_first([
            "score.regularTime.home",
            "score.regularTime.homeTeam",
        ])
        .alias("home_regular_time_goals"),

        optional_first([
            "score.regularTime.away",
            "score.regularTime.awayTeam",
        ])
        .alias("away_regular_time_goals"),


        # ---------------------------------------------------------------------
        # Extra-time score
        # ---------------------------------------------------------------------

        optional_first([
            "score.extraTime.home",
            "score.extraTime.homeTeam",
        ])
        .alias("home_extra_time_goals"),

        optional_first([
            "score.extraTime.away",
            "score.extraTime.awayTeam",
        ])
        .alias("away_extra_time_goals"),


        # ---------------------------------------------------------------------
        # Penalty shootout
        # ---------------------------------------------------------------------

        optional_first([
            "score.penalties.home",
            "score.penalties.homeTeam",
        ])
        .alias("home_penalty_goals"),

        optional_first([
            "score.penalties.away",
            "score.penalties.awayTeam",
        ])
        .alias("away_penalty_goals"),


        # ---------------------------------------------------------------------
        # Source update information
        # ---------------------------------------------------------------------

        optional_col(bronze_df, "lastUpdated", "timestamp")
        .alias("last_updated"),


        # ---------------------------------------------------------------------
        # Bronze lineage
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Normalize season and match date
# -----------------------------------------------------------------------------

# Canonical season representation uses the starting year.
# This works for competitions whose seasons span two calendar years.

silver_df = (
    silver_df
    .withColumn(
        "season_start_year",
        F.year("season_start_date"),
    )
    .withColumn(
        "match_date",
        F.to_date("match_utc_datetime"),
    )
)


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Validate competition-to-league mapping
# -----------------------------------------------------------------------------

expected_league = map_values(
    "competition_code",
    FOOTBALL_DATA_ORG_COMPETITIONS,
)

invalid_mapping_rows = (
    silver_df
    .filter(
        expected_league.isNull()
        | F.col("league_code").isNull()
        | (F.col("league_code") != expected_league)
    )
    .limit(1)
    .count()
)

if invalid_mapping_rows > 0:
    raise ValueError(
        "Competition code and canonical league mapping "
        "are inconsistent."
    )


# -----------------------------------------------------------------------------
# Deduplicate Bronze snapshots
# -----------------------------------------------------------------------------

# The same match may appear in multiple Bronze ingestions.
# Keep the most recently ingested snapshot.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "last_updated",
        "_bronze_source_file",
    ],
)



# ---------------------------------------------------------------------------
# Resolve canonical home and away team keys
# ---------------------------------------------------------------------------

team_mappings_df = load_team_mappings_by_id(
    spark,
    SILVER_SCHEMA,
    SOURCE_SYSTEM,
)

silver_df = resolve_team_keys_by_id(
    silver_df,
    team_mappings_df,
    source_id_column="home_team_id",
    output_key_column="home_team_key",
)

silver_df = resolve_team_keys_by_id(
    silver_df,
    team_mappings_df,
    source_id_column="away_team_id",
    output_key_column="away_team_key",
)

# Both sides of a match must represent different clubs.
invalid_matches = (
    silver_df
    .filter(
        F.col("home_team_key") == F.col("away_team_key")
    )
    .limit(1)
    .count()
)

if invalid_matches:
    raise ValueError(
        "Invalid matches: home and away teams resolve "
        "to the same canonical team."
    )

# Mapping must not introduce duplicate match records.
duplicate_matches = (
    silver_df
    .groupBy("match_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_matches:
    raise ValueError(
        "Duplicate match_id values after canonical resolution."
    )

print("Canonical match team resolution completed.")

# -----------------------------------------------------------------------------
# Data quality: teams must be different
# -----------------------------------------------------------------------------

invalid_team_rows = (
    silver_df
    .filter(
        F.col("home_team_id") == F.col("away_team_id")
    )
    .limit(1)
    .count()
)

if invalid_team_rows > 0:
    raise ValueError(
        "Invalid match data: home and away team IDs "
        "cannot be identical."
    )


# -----------------------------------------------------------------------------
# Data quality: goals cannot be negative
# -----------------------------------------------------------------------------

GOAL_COLUMNS = [
    "home_goals",
    "away_goals",
    "home_half_time_goals",
    "away_half_time_goals",
    "home_regular_time_goals",
    "away_regular_time_goals",
    "home_extra_time_goals",
    "away_extra_time_goals",
    "home_penalty_goals",
    "away_penalty_goals",
]

negative_goal_condition = F.lit(False)

for column_name in GOAL_COLUMNS:
    negative_goal_condition = (
        negative_goal_condition
        | (F.col(column_name) < 0)
    )

invalid_goal_rows = (
    silver_df
    .filter(negative_goal_condition)
    .limit(1)
    .count()
)

if invalid_goal_rows > 0:
    raise ValueError(
        "Invalid match data: negative goal values found."
    )


# -----------------------------------------------------------------------------
# Data quality: finished matches must have scores
# -----------------------------------------------------------------------------

invalid_finished_rows = (
    silver_df
    .filter(
        (F.col("status") == "FINISHED")
        & (
            F.col("home_goals").isNull()
            | F.col("away_goals").isNull()
        )
    )
    .limit(1)
    .count()
)

if invalid_finished_rows > 0:
    raise ValueError(
        "Finished matches without full-time scores found."
    )


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final validation
# -----------------------------------------------------------------------------

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)

if validation_metrics["row_count"] == 0:
    raise ValueError(
        "Silver matches dataset is empty. "
        "Refusing to overwrite the target table."
    )


# -----------------------------------------------------------------------------
# Write Silver
# -----------------------------------------------------------------------------

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"matches into {SILVER_TABLE}."
)
